In [1]:
import os
import gc
import torch
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftConfig, PeftModel
from tqdm import tqdm
from huggingface_hub import login
from dotenv import load_dotenv
import warnings

In [2]:
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
hf_username = "sohammandal01"

print("Loading test dataset and metrics...")
dataset = load_dataset("nrizwan/safe_ai_assignment_1", data_files={"test": "preference_test.csv"})

test_dataset = dataset['test'].select(range(100))
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

print(f"Ready to evaluate on {len(test_dataset)} prompts!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading test dataset and metrics...
Ready to evaluate on 100 prompts!


In [3]:
def evaluate_model(repo_name, is_peft=False):
    repo_id = f"{hf_username}/{repo_name}"
    print(f"\n{'='*50}")
    print(f"Evaluating: {repo_id}")
    print(f"{'='*50}")
    
    torch.cuda.empty_cache()
    gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(repo_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    if is_peft:
        config = PeftConfig.from_pretrained(repo_id)
        base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, device_map={"": 0})
        model = PeftModel.from_pretrained(base_model, repo_id, device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(repo_id, device_map={"": 0})
        
    model.eval()
    
    predictions = []
    references = []
    
    print("Generating responses...")
    for row in tqdm(test_dataset):
        prompt = f"Question: {row['Question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=30, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )
        
        generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        
        if not generated_text:
            generated_text = " "
            
        predictions.append(generated_text)
        references.append(row['More_Prefered'])

    print("\nComputing Metrics...")
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    
    formatted_references = [[ref] for ref in references]
    bleu_results = bleu_metric.compute(predictions=predictions, references=formatted_references)
    
    print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
    print(f"BLEU:    {bleu_results['bleu']:.4f}")
    
    del model
    del tokenizer

    if is_peft:
        del base_model
    
    torch.cuda.empty_cache()
    gc.collect()
    
    return {"rouge": rouge_results, "bleu": bleu_results['bleu']}

In [4]:
ppo_results = {}

ppo_models = {
    "gpt2-aligned-ppo-full-assignment": False,
    "gpt2-aligned-ppo-prefix-assignment": True,
    "gpt2-aligned-ppo-lora-assignment": True,
    "gpt2-aligned-ppo-qlora-assignment": True
}

for repo, is_peft in ppo_models.items():
    try:
        ppo_results[repo] = evaluate_model(repo, is_peft)
    except Exception as e:
        print(f"Failed to evaluate {repo}. Error: {e}")


Evaluating: sohammandal01/gpt2-aligned-ppo-full-assignment


tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: sohammandal01/gpt2-aligned-ppo-full-assignment
Key                   | Status     |  | 
----------------------+------------+--+-
v_head.summary.weight | UNEXPECTED |  | 
v_head.summary.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

Generating responses...


100%|██████████| 100/100 [00:40<00:00,  2.46it/s]



Computing Metrics...
ROUGE-1: 0.0973
ROUGE-2: 0.0235
ROUGE-L: 0.0740
BLEU:    0.0000


In [5]:
dpo_results = {}

dpo_models = {
    "gpt2-aligned-dpo-full-assignment": False,
    "gpt2-aligned-dpo-prefix-assignment": True,
    "gpt2-aligned-dpo-lora-assignment": True,
    "gpt2-aligned-dpo-qlora-assignment": True
}

for repo, is_peft in dpo_models.items():
    try:
        dpo_results[repo] = evaluate_model(repo, is_peft)
    except Exception as e:
        print(f"Failed to evaluate {repo}. Error: {e}")
        
print("\n ALL EVALUATIONS COMPLETE!")


Evaluating: sohammandal01/gpt2-aligned-dpo-full-assignment


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

Generating responses...


100%|██████████| 100/100 [00:40<00:00,  2.45it/s]



Computing Metrics...
ROUGE-1: 0.0898
ROUGE-2: 0.0194
ROUGE-L: 0.0696
BLEU:    0.0000

 ALL EVALUATIONS COMPLETE!
